← [14 - Bit-vectors](Z3-14-BitVectors-Overflow-CSharp.ipynb) | [README Z3-CSharp](README.md) | [Version Python](Z3-15-Nested-Arrays-2D-Python.ipynb) | [16 - Meal-Planner >>](Z3-16-Meal-Planner-CSharp.ipynb)

# 15. Tableaux imbriqués et grilles 2D : carrés latins, Sudoku 4×4, carré magique

Jusqu'ici, chaque variable Z3 vivait seule (`MkIntConst("x")`, `MkIntConst("y")`). Beaucoup de problèmes
combinatoires — Sudoku, carrés magiques, planification sur grille — raisonnent sur un
**tableau 2D**. Z3 offre deux encodages canoniques d'une telle grille, que ce notebook
compare :

1. **L'encodage déclaratif (idiomatique Microsoft.Z3).** Une grille `n × n` devient une liste de
   listes de variables `IntExpr`, une par case. Les contraintes (`MkDistinct` par ligne, par
   colonne, par bloc) s'écrivent en C# natif, avec des boucles. C'est l'équivalent direct de
   l'encodage `IntExpr[][]` du binding C# Microsoft.Z3, sans la couche DSL.

2. **L'encodage brut (théorie des tableaux SMT).** La grille est un `ArrayExpr` *de* `ArrayExpr`
   (`Int → (Int → Int)`) manipulé par `MkStore` / `MkSelect`. Plus verbeux, mais c'est la vraie
   théorie des tableaux de McCarthy, exposée par le binding Microsoft.Z3 sans intermédiaire.

L'objectif est de montrer comment **le même changement de paradigme** (décrire les contraintes,
laisser le solveur trouver la solution) s'étend du scalaire à la grille, et comment Microsoft.Z3
donne accès aux deux niveaux : l'élégance déclarative pour la pédagogie, et l'API brute pour
comprendre ce qui se passe sous le capot.

## Source

Le livre-ancrage de la série est **Yurichev, Dennis — *SAT/SMT by Example*** (édition publique du 12 février 2023, 666 pages) : §8.1.2 « Simple sudoku in SMT using functions and array » (p. 212) et §12.4 « Magic/Latin square of Knut Vik design » (p. 361).

Le §8.1.2 écrit un Sudoku en SMT-LIB sur un tableau `cells`, avec une opération de sélection par case et `distinct` sur les lignes, les colonnes et les blocs — la grille comme **système d'équations** que le solveur tranche. Le §12.4 résout en Z3Py les carrés de Knut Vik 5×5 et 7×7 par `Distinct` sur les lignes, les colonnes et les diagonales.

Le carnet reprend ce cadre sur des **instances plus petites** — carré latin 3×3, Sudoku 4×4, carré magique 3×3 — et y ajoute la comparaison des deux encodages : la grille comme liste de listes (`MkDistinct` par ligne, colonne et bloc) face à la vraie théorie des tableaux (`MkStore`/`MkSelect` imbriqués sur `ArrayExpr`). Le jumeau Python [`Z3-15-Nested-Arrays-2D-Python.ipynb`](Z3-15-Nested-Arrays-2D-Python.ipynb) traite les mêmes instances côté `pyz3`.

Copie archivée : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\Satisfiability and SMT Solvers\2023 - SAT_SMT_by_example.pdf` (16 456 233 octets). Voir [Z3-API/README.md — Ouvrage de référence](README.md#ouvrage-de-référence) pour l'index chapitré complet de la série.


In [1]:
#r "nuget: Microsoft.Z3"
#load "Z3NativeLoader.cs"
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;

// Bibliothèque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs).
Z3NativeLoader.Register(typeof(Context).Assembly);

// Un contexte et des helpers partagés par les cellules suivantes.
var ctx = new Context();

int Valeur(Model model, IntExpr expression)
{
    return (int)((IntNum)model.Evaluate(expression)).Int64;
}

void AfficherGrille(int[][] grille)
{
    var lignes = grille.Select(ligne => "[" + string.Join(", ", ligne.Select(v => v.ToString())) + "]");
    Console.WriteLine("[" + string.Join(", ", lignes) + "]");
}

Console.WriteLine("Imports OK : Microsoft.Z3 version " + Microsoft.Z3.Version.FullVersion + ", affichage des grilles.");

Installing Packages Microsoft.Z3

Imports OK : Microsoft.Z3 version Z3 4.12.2.0, affichage des grilles.


## 1. Carré latin 3×3 — l'approche déclarative

Un **carré latin** `n × n` contient les valeurs `1..n`, chacune apparaissant **une seule fois
par ligne et une seule fois par colonne**. C'est le squelette nu d'un Sudoku (sans les blocs).

**Encodage déclaratif** : neuf variables `IntExpr`, une par case, contraintes par des `MkDistinct`
par ligne et par colonne. Aucune boucle de backtracking à écrire : on décrit la propriété
*mathématique* de la solution, le solveur la satisfait.

In [2]:
int[][] CarreLatin3(Context contexte)
{
    var s = contexte.MkSolver();

    // Grille 3x3 : une variable IntExpr par case.
    var g = new IntExpr[3][];
    for (int r = 0; r < 3; r++)
    {
        g[r] = new IntExpr[3];
        for (int c = 0; c < 3; c++)
            g[r][c] = contexte.MkIntConst($"L_{r}_{c}");
    }

    // Domaine 1..3 + une valeur unique par ligne et par colonne.
    for (int r = 0; r < 3; r++)
    {
        for (int c = 0; c < 3; c++)
        {
            s.Add(contexte.MkGe(g[r][c], contexte.MkInt(1)));
            s.Add(contexte.MkLe(g[r][c], contexte.MkInt(3)));
        }
        s.Add(contexte.MkDistinct(g[r])); // ligne r : toutes différentes
    }

    for (int c = 0; c < 3; c++)
    {
        var colonne = new IntExpr[3];
        for (int r = 0; r < 3; r++) colonne[r] = g[r][c];
        s.Add(contexte.MkDistinct(colonne)); // colonne c : toutes différentes
    }

    if (s.Check() != Status.SATISFIABLE)
        throw new InvalidOperationException("Le carré latin 3x3 devrait être satisfiable.");

    var m = s.Model;
    var solution = new int[3][];
    for (int r = 0; r < 3; r++)
    {
        solution[r] = new int[3];
        for (int c = 0; c < 3; c++)
            solution[r][c] = Valeur(m, g[r][c]);
    }
    return solution;
}

Console.WriteLine("Carré latin 3x3 :");
AfficherGrille(CarreLatin3(ctx));

Carré latin 3x3 :


[[3, 1, 2], [2, 3, 1], [1, 2, 3]]


### Interprétation 1 — le pattern « boucles + `MkDistinct` »

Le corps de la contrainte tient en deux boucles : une pour les lignes, une pour les colonnes.
Chaque `MkDistinct([...])` est l'encodage direct de « toutes différentes ». Notez qu'**aucune
heuristic d'énumération** n'apparaît : Z3 raisonne sur les neuf variables simultanément et
produit une affectation cohérente. C'est le saut de paradigme — on ne *programme pas* la
recherche, on *déclare* la structure.

## 2. Sudoku 4×4 — le cas d'usage canonique

Le Sudoku ajoute aux contraintes du carré latin une contrainte de **bloc** : la grille 4×4
se découpe en quatre blocs 2×2, chacun devant contenir `1..4` sans répétition.

On résout deux instances : la grille **vide** (le solveur trouve une solution parmi les
960 grilles 4×4 valides) puis une grille avec **givens** (indices imposés). La seule
différence d'encodage est l'ajout de `g[r][c] == v` pour chaque donné.

In [3]:
int[][] Sudoku4x4(Context contexte, Dictionary<(int row, int col), int> givens = null)
{
    givens ??= new Dictionary<(int row, int col), int>();
    var s = contexte.MkSolver();
    var g = new IntExpr[4][];

    for (int r = 0; r < 4; r++)
    {
        g[r] = new IntExpr[4];
        for (int c = 0; c < 4; c++)
            g[r][c] = contexte.MkIntConst($"S_{r}_{c}");
    }

    // Domaine 1..4.
    for (int r = 0; r < 4; r++)
    {
        for (int c = 0; c < 4; c++)
        {
            s.Add(contexte.MkGe(g[r][c], contexte.MkInt(1)));
            s.Add(contexte.MkLe(g[r][c], contexte.MkInt(4)));
        }
        s.Add(contexte.MkDistinct(g[r])); // ligne
    }

    for (int c = 0; c < 4; c++)
    {
        var colonne = new IntExpr[4];
        for (int r = 0; r < 4; r++) colonne[r] = g[r][c];
        s.Add(contexte.MkDistinct(colonne)); // colonne
    }

    // Blocs 2x2.
    for (int br = 0; br < 4; br += 2)
    {
        for (int bc = 0; bc < 4; bc += 2)
        {
            var bloc = new IntExpr[4];
            int k = 0;
            for (int i = 0; i < 2; i++)
                for (int j = 0; j < 2; j++)
                    bloc[k++] = g[br + i][bc + j];
            s.Add(contexte.MkDistinct(bloc));
        }
    }

    // Givens (indices imposés).
    foreach (var given in givens)
        s.Add(contexte.MkEq(g[given.Key.row][given.Key.col], contexte.MkInt(given.Value)));

    if (s.Check() != Status.SATISFIABLE)
        throw new InvalidOperationException("Le Sudoku 4x4 devrait être satisfiable.");

    var m = s.Model;
    var solution = new int[4][];
    for (int r = 0; r < 4; r++)
    {
        solution[r] = new int[4];
        for (int c = 0; c < 4; c++)
            solution[r][c] = Valeur(m, g[r][c]);
    }
    return solution;
}

Console.WriteLine("Sudoku 4x4 (grille vide) :");
AfficherGrille(Sudoku4x4(ctx));

var givens = new Dictionary<(int row, int col), int>
{
    { (0, 0), 1 },
    { (1, 3), 2 },
    { (2, 1), 3 },
    { (3, 2), 4 }
};
Console.WriteLine();
Console.WriteLine("Sudoku 4x4 avec givens {(0, 0): 1, (1, 3): 2, (2, 1): 3, (3, 2): 4} :");
AfficherGrille(Sudoku4x4(ctx, givens));

Sudoku 4x4 (grille vide) :


[[4, 2, 1, 3], [3, 1, 4, 2], [1, 3, 2, 4], [2, 4, 3, 1]]


Sudoku 4x4 avec givens {(0, 0): 1, (1, 3): 2, (2, 1): 3, (3, 2): 4} :


[[1, 2, 3, 4], [3, 4, 1, 2], [4, 3, 2, 1], [2, 1, 4, 3]]


### Interprétation 2 — la grille 2D comme liste de listes

L'encodage `var g = new IntExpr[n][]` est lisible et se prête à toute
transformation C# (extraction d'un bloc par indexation de tableau, ajout d'un donné
par affectation d'égalité). Le solveur gère la combinatoire sous-jacente — ici 16 variables
sur un domaine de 4 valeurs, soit `4^16 ≈ 4 milliards` d'affectations brutes que Z3 explore
symboliquement sans énumération.

## 3. Carré magique 3×3 — sommes de lignes et colonnes

Un **carré magique** d'ordre 3 place les entiers `1..9` (tous distincts) dans une grille 3×3
telle que chaque ligne, chaque colonne et chaque diagonale somment à la **constante magique**
`M = n(n²+1)/2 = 15`. C'est l'exemple historique (le *Lo Shu*, Chine antique, ~2200 av. J.-C.).

L'encodage ajoute au carré latin une contrainte de **somme** (`MkAdd(...) == 15`) sur chaque
ligne, colonne et diagonale. La contrainte `MkDistinct` sur les neuf cases garantit l'usage
exact des entiers `1..9`.

In [4]:
int[] CarreMagique3(Context contexte)
{
    var s = contexte.MkSolver();
    var g = new IntExpr[3][];
    var plat = new IntExpr[9];
    int k = 0;

    for (int r = 0; r < 3; r++)
    {
        g[r] = new IntExpr[3];
        for (int c = 0; c < 3; c++)
        {
            g[r][c] = contexte.MkIntConst($"M_{r}_{c}");
            plat[k++] = g[r][c];
        }
    }

    foreach (var v in plat)
    {
        s.Add(contexte.MkGe(v, contexte.MkInt(1)));
        s.Add(contexte.MkLe(v, contexte.MkInt(9)));
    }
    s.Add(contexte.MkDistinct(plat)); // 9 valeurs toutes différentes

    const int MAGIE = 15;
    for (int r = 0; r < 3; r++)
        s.Add(contexte.MkEq(contexte.MkAdd(g[r]), contexte.MkInt(MAGIE))); // somme de chaque ligne

    for (int c = 0; c < 3; c++)
    {
        var colonne = new IntExpr[3];
        for (int r = 0; r < 3; r++) colonne[r] = g[r][c];
        s.Add(contexte.MkEq(contexte.MkAdd(colonne), contexte.MkInt(MAGIE))); // somme de chaque colonne
    }

    var diagonalePrincipale = new IntExpr[3];
    var diagonaleOpposee = new IntExpr[3];
    for (int i = 0; i < 3; i++)
    {
        diagonalePrincipale[i] = g[i][i];
        diagonaleOpposee[i] = g[i][2 - i];
    }
    s.Add(contexte.MkEq(contexte.MkAdd(diagonalePrincipale), contexte.MkInt(MAGIE)));
    s.Add(contexte.MkEq(contexte.MkAdd(diagonaleOpposee), contexte.MkInt(MAGIE)));

    if (s.Check() != Status.SATISFIABLE)
        throw new InvalidOperationException("Le carré magique 3x3 devrait être satisfiable.");

    var m = s.Model;
    var solution = new int[3][];
    for (int r = 0; r < 3; r++)
    {
        solution[r] = new int[3];
        for (int c = 0; c < 3; c++)
            solution[r][c] = Valeur(m, g[r][c]);
    }
    return solution.SelectMany(ligne => ligne).ToArray();
}

var sol = CarreMagique3(ctx);
Console.WriteLine("Carré magique 3x3 (constante = 15) :");
AfficherGrille(new[]
{
    new[] { sol[0], sol[1], sol[2] },
    new[] { sol[3], sol[4], sol[5] },
    new[] { sol[6], sol[7], sol[8] }
});

// Vérification : la solution satisfait bien toutes les contraintes de somme.
Console.WriteLine("Sommes lignes   : [" + string.Join(", ", new[]
{
    sol[0] + sol[1] + sol[2],
    sol[3] + sol[4] + sol[5],
    sol[6] + sol[7] + sol[8]
}) + "]");
Console.WriteLine("Sommes colonnes : [" + string.Join(", ", new[]
{
    sol[0] + sol[3] + sol[6],
    sol[1] + sol[4] + sol[7],
    sol[2] + sol[5] + sol[8]
}) + "]");

Carré magique 3x3 (constante = 15) :


[[4, 3, 8], [9, 5, 1], [2, 7, 6]]


Sommes lignes   : [15, 15, 15]


Sommes colonnes : [15, 15, 15]


### Interprétation 3 — `MkAdd` comme contrainte linéaire

`MkAdd(...) == 15` est une **contrainte linéaire** entière que le solveur traite au même
titre qu'une inégalité. Le carré magique combine donc trois familles de contraintes —
domaine (`1..9`), **toutes différentes** (`MkDistinct`), **somme exacte** (`MkAdd`) — dans un
même solveur. La solution trouvée est le *Lo Shu* classique (à rotation/symétrie près,
puisque le solveur renvoie la première solution sat).

## 4. L'API brute — `Array` de `Array` (`Store` / `Select` imbriqués)

L'encodage déclaratif cache la mécanique : il n'y a pas de « vrai » tableau Z3, seulement
des variables scalaires nommées `L_0_0`, `L_0_1`… La **théorie des tableaux** de Z3
(axiomes de McCarthy : `select(store(a, i, v), i) == v` et `select(store(a, i, v), j) == select(a, j)`
si `i != j`) offre un véritable tableau symbolique via `ArrayExpr`.

Pour une grille 2D, on emboîte deux `ArrayExpr` : un **tableau externe** `Int → (Int → Int)`
qui mappe l'indice de ligne à un **tableau interne** `Int → Int` (la ligne). La cellule
`(i, j)` se lit `MkSelect(MkSelect(grid, i), j)` et s'écrit par `MkStore` imbriqué. L'exemple
construit une grille 2×2 `[[1, 2], [3, 4]]` et relit chaque cellule.

In [5]:
int[][] Grille2DRaw(Context contexte)
{
    var s = contexte.MkSolver();
    var sortInt = contexte.MkIntSort();
    var sortLigne = contexte.MkArraySort(sortInt, sortInt);   // Int -> Int
    var sortGrille = contexte.MkArraySort(sortInt, sortLigne); // Int -> (Int -> Int)

    // Tableau interne constant (tout index -> 0) : equivalent C# de K(sort, valeur).
    ArrayExpr zero = (ArrayExpr)contexte.MkConstArray(sortInt, contexte.MkInt(0));

    // Construction de chaque ligne par Store successifs.
    ArrayExpr ligne0 = (ArrayExpr)contexte.MkStore(
        (ArrayExpr)contexte.MkStore(zero, contexte.MkInt(0), contexte.MkInt(1)),
        contexte.MkInt(1), contexte.MkInt(2));   // ligne 0 = [1, 2, 0, 0, ...]
    ArrayExpr ligne1 = (ArrayExpr)contexte.MkStore(
        (ArrayExpr)contexte.MkStore(zero, contexte.MkInt(0), contexte.MkInt(3)),
        contexte.MkInt(1), contexte.MkInt(4));   // ligne 1 = [3, 4, 0, 0, ...]

    // Grille externe : Int -> (Int -> Int). On stocke chaque ligne a son indice.
    ArrayExpr grille = (ArrayExpr)contexte.MkStore(
        (ArrayExpr)contexte.MkStore(
            (ArrayExpr)contexte.MkConstArray(sortInt, zero),
            contexte.MkInt(0), ligne0),
        contexte.MkInt(1), ligne1);

    // Lecture d'une cellule : Select imbrique (le Select externe rend un tableau interne).
    IntExpr Cellule(ArrayExpr g, int i, int j)
    {
        var ligne = (ArrayExpr)contexte.MkSelect(g, contexte.MkInt(i));
        return (IntExpr)contexte.MkSelect(ligne, contexte.MkInt(j));
    }

    // Verification : lecture de chaque cellule par Select imbrique.
    s.Add(contexte.MkEq(Cellule(grille, 0, 0), contexte.MkInt(1)));
    s.Add(contexte.MkEq(Cellule(grille, 0, 1), contexte.MkInt(2)));
    s.Add(contexte.MkEq(Cellule(grille, 1, 0), contexte.MkInt(3)));
    s.Add(contexte.MkEq(Cellule(grille, 1, 1), contexte.MkInt(4)));

    if (s.Check() != Status.SATISFIABLE)
        throw new InvalidOperationException("La grille 2D devrait etre satisfiable.");

    var m = s.Model;
    var solution = new int[2][];
    for (int i = 0; i < 2; i++)
    {
        solution[i] = new int[2];
        for (int j = 0; j < 2; j++)
            solution[i][j] = (int)((IntNum)m.Evaluate(Cellule(grille, i, j))).Int64;
    }
    return solution;
}

Console.WriteLine("Grille 2x2 (Array de Array, Store/Select) :");
AfficherGrille(Grille2DRaw(ctx));

Grille 2x2 (Array de Array, Store/Select) :


[[1, 2], [3, 4]]


### Interprétation 4 — déclaratif vs brut

Les deux encodages produisent la même grille, mais ils ne servent pas le même but :

| Aspect | Déclaratif (listes de `IntExpr`) | Brut (`ArrayExpr` de `ArrayExpr`) |
|--------|------------------------------|---------------------------|
| **Lisibilité** | C# natif, tableaux et boucles | `MkStore`/`MkSelect` emboîtés, verbeux |
| **Sémantique** | Variables scalaires liées par contraintes | Vrai tableau symbolique de McCarthy |
| **Cas d'usage** | Pédagogie, Sudoku, puzzles | Vérification de programmes, état mutable |
| **Quand l'utiliser** | Toujours pour modéliser un puzzle | Dès qu'on raisonne sur un *tableau en tant que tel* (mise à jour en place, preuve de tri, etc.) |

Le brut est l'outil de choix pour la **vérification de programmes** (prouver qu'un tri modifie
un tableau comme attendu) ; le déclaratif est l'outil de choix pour la **modélisation de
puzzles**. Microsoft.Z3 expose les deux sans intermédiaire — là où le binding C# Z3.Linq masque le
brut derrière son DSL `Theorem<Grid>`.

## 5. Exercices

Les trois exercices suivants étendent les modèles ci-dessus. Chaque stub est à compléter :
votre code doit produire une solution vérifiée (le solveur la satisfait par construction).

### Exercice 1 — Carré latin 3×3 avec contraintes diagonales

Reprenez le modèle du carré latin 3×3 (cellule §1) et ajoutez la contrainte que les **deux
diagonales** sont aussi « toutes différentes ». Combien de solutions reste-t-il ? Affichez-en
une.

*Indice* : `MkDistinct` sur `g[i][i]` et sur `g[i][2-i]`.
Pour compter les solutions, ajoutez une contrainte bloquante (`MkOr(...)`) à chaque tour
en niant le modèle trouvé.

In [6]:
// Exercice 1 : carré latin 3x3 avec diagonales aussi toutes différentes.
// TODO etudiant :
// Etape 1 : reprendre la grille 3x3 et les contraintes ligne/colonne de la cellule §1.
// Etape 2 : ajouter MkDistinct sur la diagonale principale et la diagonale opposée.
// Etape 3 : afficher une solution.
//
// Indice : MkDistinct sur g[i][i] et MkDistinct sur g[i][2-i]
void Exercice1CarreLatinDiagonales()
{
    // TODO etudiant : implémentez la résolution avec les deux diagonales distinctes
    Console.WriteLine("Exercice 1 à compléter : carré latin 3x3 avec contraintes diagonales.");
}

Exercice1CarreLatinDiagonales();

Exercice 1 à compléter : carré latin 3x3 avec contraintes diagonales.


### Exercice 2 — Sudoku 4×4 avec une autre grille de départ

Reprenez `Sudoku4x4` et résolvez avec une grille de départ **différente** et plus contraignante
(par exemple 5 givens bien choisis). Vérifiez que la solution respecte tous les givens.

*Indice* : passez un dictionnaire `givens` plus fourni à `Sudoku4x4`, puis assertez que chaque
solution renvoyée contient bien la valeur imposée à chaque position donnée.

In [7]:
// Exercice 2 : Sudoku 4x4 avec une grille de départ différente (>= 5 givens).
// TODO etudiant :
// Etape 1 : choisir 5 givens cohérents (qui n'épuisent aucune ligne/colonne/bloc).
// Etape 2 : appeler Sudoku4x4(givens) et afficher la solution.
// Etape 3 : vérifier que la solution respecte chaque donné (assert par assert).
void Exercice2Sudoku4x4Givens()
{
    // TODO etudiant : implémentez la résolution avec une autre grille de départ
    Console.WriteLine("Exercice 2 à compléter : Sudoku 4x4 avec une autre grille de départ.");
}

Exercice2Sudoku4x4Givens();

Exercice 2 à compléter : Sudoku 4x4 avec une autre grille de départ.


### Exercice 3 — Vérificateur de carré magique

Écrivez un vérificateur qui, étant donné une grille 3×3 **concrète** (par exemple
`[[2,7,6],[9,5,1],[4,3,8]]`), utilise Z3 pour **prouver** qu'elle est magique (ou exhiber la
contrainte violée). On fixe les variables aux valeurs de la grille puis on vérifie que toutes
les contraintes de somme (cellule §3) sont satisfaites.

*Indice* : `s.Add(contexte.MkEq(g[r][c], contexte.MkInt(valeur[r][c])))` pour figer la grille, puis `s.Check() == Status.SATISFIABLE`
prouve que toutes les contraintes de somme tiennent.

In [8]:
// Exercice 3 : vérifier qu'une grille 3x3 concrète est un carré magique.
// TODO etudiant :
// Etape 1 : figer g[r][c] aux valeurs d'une grille candidate (ex. [[2,7,6],[9,5,1],[4,3,8]]).
// Etape 2 : ajouter les contraintes de somme (lignes, colonnes, diagonales = 15).
// Etape 3 : s.Check() == Status.SATISFIABLE prouve que la grille est magique ; sinon, elle ne l'est pas.
string Exercice3VerificateurCarreMagique(int[][] candidate)
{
    // TODO etudiant : implémentez la preuve de caractère magique par Z3
    return "à compléter";
}

Console.WriteLine("Exercice 3 à compléter : vérificateur de carré magique.");
Console.WriteLine("Verdict Exercice 3 : " + Exercice3VerificateurCarreMagique(new[]
{
    new[] { 2, 7, 6 },
    new[] { 9, 5, 1 },
    new[] { 4, 3, 8 }
}));

Exercice 3 à compléter : vérificateur de carré magique.


Verdict Exercice 3 : à compléter


## Conclusion

Ce notebook a étendu la modélisation Z3 du **scalaire** à la **grille 2D**, via deux
encodages complémentaires :

- **Déclaratif** (listes de `IntExpr` + `MkDistinct`/`MkAdd`) : l'outil de choix pour modéliser
  puzzles et problèmes combinatoires sur grille — carré latin, Sudoku, carré magique.
- **Brut** (`ArrayExpr` de `ArrayExpr`, `MkStore`/`MkSelect`) : la théorie des tableaux de McCarthy,
  indispensable pour raisonner sur un tableau en tant qu'objet mutable (vérification de tri,
  preuve de programmes).

Le même changement de paradigme — *décrire les contraintes, pas l'algorithme* — passe du
scalaire à la grille sans rupture de style : on ajoute des dimensions aux boucles et aux
tableaux C#, et le solveur gère la combinatoire. C'est précisément ce que rend visible l'API
complète de `Microsoft.Z3`, sans la couche déclarative du binding C# Z3.Linq.

**Pour aller plus loin** : combinez l'encodage grille avec `Optimize` (cellule §6 du notebook
06) pour des variantes optimisées — par exemple le carré magique **anti-magique** (sommes
toutes différentes) ou un Sudoku avec objectif de minimiser le nombre de givens.

← [14 - Bit-vectors](Z3-14-BitVectors-Overflow-CSharp.ipynb) | [README Z3-CSharp](README.md) | [16 - Meal-Planner >>](Z3-16-Meal-Planner-CSharp.ipynb)